# LeetCode 644 — Maximum Average Subarray II bằng Python

- Link: https://leetcode.com/problems/maximum-average-subarray-ii/
- Độ khó: **Hard** (bài **Premium**)
- Chủ đề: Tìm kiếm nhị phân trên số thực, Tổng tiền tố (prefix sum), Bao lồi (convex hull), Phương pháp Dinkelbach, Phân số

> **Ghi chú Premium:** đây là bài dành cho tài khoản LeetCode Premium. Phần đề bài dưới đây được viết lại theo hiểu biết về bài (chữ ký hàm, ví dụ, ràng buộc, sai số cho phép `10^-5`). Nếu bạn có tài khoản Premium, hãy **đối chiếu lại** chi tiết đề trước khi nộp.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findMaxAverage(self, nums: List[int], k: int) -> float:

```

## 1. Giới thiệu & mục tiêu học

Ở bài dễ (LC 643) độ dài đoạn **cố định** bằng `k` — cửa sổ trượt là xong. Bài này cho phép độ dài **từ `k` trở lên**, và trung bình cộng **không** "cộng dồn" được như tổng: đoạn có trung bình lớn nhất không nhất thiết chứa đoạn tốt nhất ngắn hơn. Cú lật quan trọng là kỹ thuật **"tìm kiếm nhị phân trên đáp án + trừ đi giá trị đoán"**: hỏi "có đoạn nào trung bình `>= x` không?" biến thành "có đoạn nào **tổng** của `nums[i] - x` không âm không?" — câu hỏi về tổng thì dễ.

Sau notebook này bạn sẽ:

1. Viết brute force `O(n^2)` bằng tổng tiền tố và hiểu vì sao nó quá chậm.
2. Nắm mẹo **trừ trung bình đoán**: `avg >= x ⇔ Σ(a_i - x) >= 0`, rồi kiểm tra bằng "tổng tiền tố trừ min tiền tố" trong `O(n)`.
3. Viết binary search trên số thực đúng chuẩn (điều kiện dừng theo sai số, số vòng lặp).
4. Biết hai lời giải **chính xác** (không sai số): bao lồi các điểm tiền tố và phương pháp **Dinkelbach**.
5. Kiểm chứng bằng brute force dùng `Fraction` (phân số chính xác) trên input ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng số nguyên `nums` gồm `n` phần tử và số nguyên `k`. Tìm **đoạn con liên tiếp** có độ dài **ít nhất `k`** sao cho **trung bình cộng** lớn nhất; trả về giá trị trung bình đó. Đáp án có sai số nhỏ hơn `10^-5` được chấp nhận.

**Ví dụ 1**

```text
Input:  nums = [1, 12, -5, -6, 50, 3], k = 4
Output: 12.75
Giải thích:
  - độ dài 4: các trung bình 0.5, 12.75, 10.5        -> lớn nhất 12.75 ([12,-5,-6,50])
  - độ dài 5: các trung bình 10.4, 10.8
  - độ dài 6: trung bình 9.16667
  Lớn nhất là 12.75.
```

**Ví dụ 2**

```text
Input:  nums = [5], k = 1
Output: 5.0
```

**Ràng buộc (theo hiểu biết về bài)**

- `n == nums.length`, `1 <= k <= n <= 10^4`
- `-10^4 <= nums[i] <= 10^4`

In [1]:
nums1, k1 = [1, 12, -5, -6, 50, 3], 4
n = len(nums1)
for L in range(k1, n + 1):
    avgs = [sum(nums1[i:i + L]) / L for i in range(n - L + 1)]
    print(f"độ dài {L}: " + ", ".join(f"{a:.5f}" for a in avgs))

độ dài 4: 0.50000, 12.75000, 10.50000
độ dài 5: 10.40000, 10.80000
độ dài 6: 9.16667


## 3. Kiến thức nền

### 3.1. Tổng tiền tố

`S[0] = 0`, `S[i] = nums[0] + ... + nums[i-1]`. Tổng đoạn `nums[i..j-1]` = `S[j] - S[i]`, độ dài `j - i`. Trung bình = `(S[j] - S[i]) / (j - i)`.

In [2]:
from itertools import accumulate
S = [0] + list(accumulate(nums1))
print("S =", S)
print("tổng nums[1..4] =", S[5] - S[1], "| trung bình =", (S[5] - S[1]) / 4)

S = [0, 1, 13, 8, 2, 52, 55]
tổng nums[1..4] = 51 | trung bình = 12.75


### 3.2. `fractions.Fraction` — phân số chính xác

Số thực dấu phẩy động có sai số làm tròn; so sánh hai trung bình rất gần nhau có thể sai. `Fraction(a, b)` lưu tử/mẫu nguyên → so sánh **chính xác**. Dùng cho brute force "trọng tài".

In [3]:
from fractions import Fraction
print(0.1 + 0.2 == 0.3, Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))
print(Fraction(51, 4), float(Fraction(51, 4)), max(Fraction(1, 3), Fraction(33, 100)))

False True
51/4 12.75 1/3


### 3.3. So sánh hai phân số bằng nhân chéo

`a/b >= c/d` (với `b, d > 0`) ⇔ `a·d >= c·b`. Không cần chia, không sai số — dùng trong lời giải bao lồi.

In [4]:
def frac_ge(a, b, c, d):
    return a * d >= c * b

print(frac_ge(51, 4, 21, 2), frac_ge(21, 2, 51, 4))   # 12.75 >= 10.5 ?

True False


### 3.4. Tìm kiếm nhị phân trên số thực

Không có "phần tử liền sau" như số nguyên, nên ta lặp tới khi `hi - lo < eps` (hoặc lặp một số vòng cố định). Mỗi vòng thu hẹp khoảng một nửa → số vòng ≈ `log2((hi - lo) / eps)`.

In [5]:
import math
lo, hi, eps = 0.0, 2.0, 1e-7
it = 0
while hi - lo > eps:
    mid = (lo + hi) / 2
    if mid * mid >= 2:
        hi = mid
    else:
        lo = mid
    it += 1
print(f"sqrt(2) ≈ {lo:.7f} sau {it} vòng; log2(2/1e-7) ≈ {math.log2(2 / eps):.1f}")

sqrt(2) ≈ 1.4142135 sau 25 vòng; log2(2/1e-7) ≈ 24.3


### 3.5. Tổng đoạn lớn nhất khi độ dài `>= k` (không có trung bình)

Bài phụ: tìm `max (S[j] - S[i])` với `j - i >= k`. Duyệt `j` từ `k` tới `n`, giữ `min_prefix = min(S[0..j-k])`. Đáp án `max(S[j] - min_prefix)`. `O(n)`.

In [6]:
def max_sum_len_at_least_k(a, k):
    S = [0] + list(accumulate(a))
    best = float('-inf')
    min_prefix = float('inf')
    for j in range(k, len(a) + 1):
        min_prefix = min(min_prefix, S[j - k])
        best = max(best, S[j] - min_prefix)
    return best

print(max_sum_len_at_least_k([1, 12, -5, -6, 50, 3], 4), max_sum_len_at_least_k([-3, -1, -2], 2))

55 -3


### 3.6. Tích có hướng (cross product) và bao lồi dưới

Với ba điểm `O, A, B`: `cross(O, A, B) = (A.x - O.x)(B.y - O.y) - (A.y - O.y)(B.x - O.x)`. Dương ⇔ `O → A → B` rẽ trái. Bao lồi **dưới** của các điểm thêm theo `x` tăng dần: khi thêm điểm mới, bỏ điểm cuối khi ba điểm cuối **không** rẽ trái.

In [7]:
def cross(o, a, b):
    return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

pts = [(0, 0), (1, 3), (2, 1), (3, 2), (4, 0), (5, 4)]
hull = []
for p in pts:
    while len(hull) >= 2 and cross(hull[-2], hull[-1], p) <= 0:
        hull.pop()
    hull.append(p)
print("bao lồi dưới:", hull)

bao lồi dưới: [(0, 0), (4, 0), (5, 4)]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Vì sao không tham lam được

Trung bình không "cộng dồn": nối thêm một phần tử lớn có thể kéo trung bình lên, nối thêm phần tử nhỏ kéo xuống, và đoạn tối ưu độ dài `>= k` không suy ra trực tiếp từ đoạn tối ưu độ dài `k`. Ví dụ `nums = [0, 0, 10, 0, 9, 9]`, `k = 2`: đoạn tốt nhất dài 2 là `[9, 9]` (9.0), nhưng nếu đổi thành `[0, 0, 10, 9, 0, 0]` thì `[10, 9]` (9.5) — không có quy luật đơn giản.

### 4.2. Mẹo trừ trung bình đoán

Câu hỏi quyết định: "**có** đoạn độ dài `>= k` với trung bình `>= x` không?"

```text
(a_i + ... + a_j) / L >= x   ⇔   (a_i - x) + ... + (a_j - x) >= 0
```

Vậy chỉ cần kiểm tra: mảng `b_i = a_i - x` có đoạn độ dài `>= k` với **tổng `>= 0`** không — chính là bài phụ 3.5, `O(n)`.

Câu trả lời **đơn điệu**: nếu có đoạn trung bình `>= x` thì cũng có đoạn trung bình `>= x'` với mọi `x' < x`. → binary search trên `x`.

### 4.3. Tính tay `check(x)` với ví dụ 1

`x = 12`: `b = [-11, 0, -17, -18, 38, -9]`, tiền tố `B = [0, -11, -11, -28, -46, -8, -17]`.

| j | B[j] | min(B[0..j-4]) | B[j] - min | `>= 0`? |
|---|---|---|---|---|
| 4 | -46 | B[0] = 0 | -46 | không |
| 5 | -8 | min(0, -11) = -11 | 3 | **có** |

→ có đoạn trung bình `>= 12` (đoạn `nums[1..4]`, trung bình 12.75). Với `x = 13`, mọi giá trị đều âm → không có.

In [8]:
def check_verbose(nums, k, x):
    b = [a - x for a in nums]
    B = [0] + list(accumulate(b))
    min_prefix = float('inf')
    print(f"x = {x}: b = {b}")
    for j in range(k, len(nums) + 1):
        min_prefix = min(min_prefix, B[j - k])
        print(f"  j={j}: B[j]={B[j]:>6}, min tiền tố={min_prefix:>6}, hiệu={B[j] - min_prefix:>6}")
        if B[j] - min_prefix >= 0:
            return True
    return False

print("check(12):", check_verbose(nums1, k1, 12))
print("check(13):", check_verbose(nums1, k1, 13))

x = 12: b = [-11, 0, -17, -18, 38, -9]
  j=4: B[j]=   -46, min tiền tố=     0, hiệu=   -46
  j=5: B[j]=    -8, min tiền tố=   -11, hiệu=     3
check(12): True
x = 13: b = [-12, -1, -18, -19, 37, -10]
  j=4: B[j]=   -50, min tiền tố=     0, hiệu=   -50
  j=5: B[j]=   -13, min tiền tố=   -12, hiệu=    -1
  j=6: B[j]=   -23, min tiền tố=   -13, hiệu=   -10
check(13): False


## 5. Cách 1 — Brute Force: mọi đoạn bằng tổng tiền tố

**Ý tưởng.** Với mọi cặp `i < j` có `j - i >= k`, tính `(S[j] - S[i]) / (j - i)`, lấy max. Bản "trọng tài" dùng `Fraction` để so sánh chính xác.

In [9]:
def max_avg_brute_exact(nums, k):
    S = [0] + list(accumulate(nums))
    n = len(nums)
    return max(Fraction(S[j] - S[i], j - i) for i in range(n) for j in range(i + k, n + 1))

def max_avg_brute(nums, k):
    S = [0] + list(accumulate(nums))
    n = len(nums)
    best = float('-inf')
    for i in range(n):
        for j in range(i + k, n + 1):
            avg = (S[j] - S[i]) / (j - i)
            if avg > best:
                best = avg
    return best

print(max_avg_brute(nums1, k1), max_avg_brute_exact(nums1, k1), max_avg_brute([5], 1))
assert max_avg_brute_exact(nums1, k1) == Fraction(51, 4)
assert abs(max_avg_brute([5], 1) - 5.0) < 1e-9

12.75 51/4 5.0


**Độ phức tạp.** `O(n^2)` cặp → `n = 10^4` cho `5·10^7` phép chia — vài chục giây trong Python. Space `O(n)`.

**Nút thắt.** Ta đi tìm **giá trị lớn nhất của một tỉ số** trên `O(n^2)` cặp. Tỉ số khó tối ưu trực tiếp; nhưng câu hỏi "tỉ số `>= x`?" chuyển thành câu hỏi về **tổng** — tối ưu được trong `O(n)`.

In [10]:
import time, random
random.seed(0)
for n_ in (500, 1000, 2000):
    arr = [random.randint(-10**4, 10**4) for _ in range(n_)]
    t = time.perf_counter()
    max_avg_brute(arr, n_ // 10)
    print(f"n={n_}: {time.perf_counter() - t:.3f}s")

n=500: 0.009s
n=1000: 0.037s
n=2000: 0.082s


## 6. Bước đột phá

**Insight 1 (tham số hóa).** Đặt `F(x) = max` trên các đoạn độ dài `>= k` của `Σ (a_i - x)`. Khi đó:
- `F(x) >= 0` ⇔ có đoạn trung bình `>= x`.
- `F` **giảm dần** theo `x` → đáp án `x*` là điểm mà `F` đổi dấu.

**Insight 2 (kiểm tra `O(n)`).** `F(x) >= 0` ⇔ tồn tại `j >= k` với `B[j] - min(B[0..j-k]) >= 0`.

**Insight 3 (chính xác bằng hình học).** Trung bình đoạn `[i, j)` = **hệ số góc** đường nối hai điểm `P_i = (i, S[i])` và `P_j = (j, S[j])`. Bài toán = tìm cặp điểm cách nhau `>= k` trên trục `x` có hệ số góc lớn nhất. Với mỗi `j`, ứng viên tốt nhất nằm trên **bao lồi dưới** của `P_0..P_{j-k}`.

**Invariant của binary search:** luôn có `check(lo) = True` (tồn tại đoạn trung bình `>= lo`) và đáp án `< hi` hoặc `<= hi`.

**Quy tắc nếu…thì…:**
- Nếu `check(mid)` đúng thì đáp án `>= mid` → `lo = mid`.
- Nếu sai thì đáp án `< mid` → `hi = mid`.
- Dừng khi `hi - lo < 10^-5` (dùng `10^-6` cho an toàn); trả `lo`.

## 7. Các cách tiếp theo

Notebook trình bày **4 cách**:

| # | Cách | Time | Kết quả |
|---|---|---|---|
| 1 | Brute force mọi đoạn (mục 5) | `O(n^2)` | chính xác |
| 2 | Binary search trên đáp án + kiểm tra `O(n)` | `O(n log(R/ε))` | gần đúng (sai số `ε`) |
| 3 | Bao lồi dưới các điểm tiền tố | `O(n log n)` | chính xác |
| 4 | Dinkelbach (lặp tỉ số) | `O(n · số vòng)`, thường vài vòng | chính xác |

(`R` = `max(nums) - min(nums)` ≤ `2·10^4`.)

### 7.1. Cách 2 — Binary search trên đáp án

**Giải thích.** `lo = min(nums)`, `hi = max(nums)` (trung bình mọi đoạn nằm trong khoảng này). Lặp: `mid = (lo + hi) / 2`; nếu `check(mid)` thì `lo = mid`, ngược lại `hi = mid`.

`check(x)` cài đặt **không tạo mảng mới**: giữ tổng tiền tố `cur` của `b` tới `j`, tổng tiền tố `prev` tới `j - k`, và `min_prev`.

In [11]:
def check(nums, k, x):
    cur = 0.0
    for i in range(k):
        cur += nums[i] - x
    if cur >= 0:
        return True
    prev = 0.0
    min_prev = 0.0
    for j in range(k, len(nums)):
        cur += nums[j] - x                 # B[j+1]
        prev += nums[j - k] - x            # B[j+1-k]
        if prev < min_prev:
            min_prev = prev
        if cur - min_prev >= 0:
            return True
    return False

def max_avg_binary(nums, k, eps=1e-6):
    lo, hi = float(min(nums)), float(max(nums))
    while hi - lo > eps:
        mid = (lo + hi) / 2
        if check(nums, k, mid):
            lo = mid
        else:
            hi = mid
    return lo

r = max_avg_binary(nums1, k1)
print(r, max_avg_binary([5], 1))
assert abs(r - 12.75) < 1e-5

12.749999403953552 5.0


**Đọc code:**

- Vòng đầu tính `B[k]` = tổng `k` phần tử đầu của `b` — đoạn `[0, k)`.
- Trong vòng chính, `cur` là `B[j+1]`, `prev` là `B[j+1-k]`; `min_prev` = `min(B[0..j+1-k])` (khởi tạo `B[0] = 0`).
- `cur - min_prev >= 0` ⇔ có đoạn kết thúc tại `j` (độ dài `>= k`) với tổng `b` không âm.
- `lo = min(nums)` hợp lệ vì `check(min(nums))` luôn đúng (mọi đoạn có trung bình `>= min`).

**Số vòng lặp:** `log2(2·10^4 / 10^-6) ≈ 34` → `34 · 10^4 = 3,4·10^5` phép — nhanh.

**Code nộp LeetCode (Cách 2):**

In [12]:
from typing import List

class SolutionBinary:
    def findMaxAverage(self, nums: List[int], k: int) -> float:
        def ok(x: float) -> bool:
            cur = sum(nums[:k]) - x * k
            if cur >= 0:
                return True
            prev = min_prev = 0.0
            for j in range(k, len(nums)):
                cur += nums[j] - x
                prev += nums[j - k] - x
                if prev < min_prev:
                    min_prev = prev
                if cur >= min_prev:
                    return True
            return False

        lo, hi = float(min(nums)), float(max(nums))
        while hi - lo > 1e-6:
            mid = (lo + hi) / 2
            if ok(mid):
                lo = mid
            else:
                hi = mid
        return lo

assert abs(SolutionBinary().findMaxAverage(nums1, k1) - 12.75) < 1e-5
print("SolutionBinary OK")

SolutionBinary OK


**Độ phức tạp Cách 2:** Time `O(n · log(R / ε))`, Space `O(1)`.

### 7.2. Cách 3 — Bao lồi dưới (chính xác)

**Giải thích.** Điểm `P_i = (i, S[i])`. Trung bình đoạn `[i, j)` = hệ số góc `P_i P_j`. Với `j` cố định, cần `i <= j - k` sao cho hệ số góc `P_i → P_j` lớn nhất.

- Điểm `P_i` nằm **phía trên** bao lồi dưới thì không bao giờ tốt nhất: đường từ `P_j` (bên phải) chạm bao lồi dưới "từ bên dưới" luôn có hệ số góc lớn hơn hoặc bằng.
- Trên bao lồi dưới, hệ số góc tới `P_j` **tăng rồi giảm** theo thứ tự điểm → tìm đỉnh bằng binary search.

```text
hull = []
với j từ k tới n:
    thêm P_{j-k} vào hull (giữ tính lồi dưới)
    tìm điểm trên hull cho hệ số góc lớn nhất tới P_j (binary search)
    cập nhật đáp án (so sánh phân số bằng nhân chéo)
```

In [13]:
def max_avg_hull(nums, k):
    S = [0] + list(accumulate(nums))
    n = len(nums)
    hull = []                                  # chỉ số i, điểm (i, S[i])
    best_num, best_den = None, None

    def slope_ge(i1, i2, j):
        """hệ số góc P_i1->P_j >= hệ số góc P_i2->P_j ?"""
        return (S[j] - S[i1]) * (j - i2) >= (S[j] - S[i2]) * (j - i1)

    for j in range(k, n + 1):
        p = j - k
        while len(hull) >= 2:
            a, b = hull[-2], hull[-1]
            # cross((a,S[a]), (b,S[b]), (p,S[p])) <= 0 -> b không thuộc bao lồi dưới
            if (b - a) * (S[p] - S[a]) - (S[b] - S[a]) * (p - a) <= 0:
                hull.pop()
            else:
                break
        hull.append(p)
        lo, hi = 0, len(hull) - 1
        while lo < hi:
            m = (lo + hi) // 2
            if slope_ge(hull[m + 1], hull[m], j):
                lo = m + 1
            else:
                hi = m
        i = hull[lo]
        num, den = S[j] - S[i], j - i
        if best_num is None or num * best_den > best_num * den:
            best_num, best_den = num, den
    return Fraction(best_num, best_den)

print(max_avg_hull(nums1, k1), max_avg_hull([5], 1))
assert max_avg_hull(nums1, k1) == Fraction(51, 4)

51/4 5


**Đọc code:**

- `hull` lưu **chỉ số** `i`; tọa độ điểm là `(i, S[i])` — toàn số nguyên, mọi phép so sánh đều chính xác.
- Điều kiện bỏ điểm: tích có hướng `<= 0` (không rẽ trái) — điểm giữa nằm trên hoặc phía trên đoạn nối hai điểm kia.
- Binary search: nếu điểm `m+1` cho hệ số góc `>=` điểm `m` thì đỉnh ở bên phải.
- Kết quả trả về là `Fraction` chính xác; LeetCode cần `float` → `float(...)`.

**Code nộp LeetCode (Cách 3):**

In [14]:
class SolutionHull:
    def findMaxAverage(self, nums: List[int], k: int) -> float:
        S = [0]
        for x in nums:
            S.append(S[-1] + x)
        n = len(nums)
        hull = []
        bn, bd = -10**9, 1
        for j in range(k, n + 1):
            p = j - k
            while len(hull) >= 2:
                a, b = hull[-2], hull[-1]
                if (b - a) * (S[p] - S[a]) - (S[b] - S[a]) * (p - a) <= 0:
                    hull.pop()
                else:
                    break
            hull.append(p)
            lo, hi = 0, len(hull) - 1
            Sj = S[j]
            while lo < hi:
                m = (lo + hi) // 2
                i1, i2 = hull[m + 1], hull[m]
                if (Sj - S[i1]) * (j - i2) >= (Sj - S[i2]) * (j - i1):
                    lo = m + 1
                else:
                    hi = m
            i = hull[lo]
            num, den = Sj - S[i], j - i
            if num * bd > bn * den:
                bn, bd = num, den
        return bn / bd

assert SolutionHull().findMaxAverage(nums1, k1) == 12.75
print("SolutionHull OK")

SolutionHull OK


**Độ phức tạp Cách 3:** mỗi điểm vào/ra bao lồi tối đa một lần (`O(n)` tổng), mỗi truy vấn binary search `O(log n)` → Time **`O(n log n)`**, Space `O(n)`.

### 7.3. Cách 4 — Phương pháp Dinkelbach (lặp tỉ số)

**Giải thích.** Dinkelbach là kỹ thuật tổng quát cho bài "**tối đa hóa tỉ số**" `A(đoạn) / B(đoạn)`:

```text
λ = trung bình một đoạn bất kỳ hợp lệ (ví dụ k phần tử đầu)
lặp:
    tìm đoạn D tối đa hóa Σ(a_i - λ) với độ dài >= k     (bài phụ 3.5, O(n))
    nếu giá trị tối đa == 0: λ là đáp án, dừng
    λ = trung bình của D                                  (luôn lớn hơn λ cũ)
```

Thay vì chia đôi khoảng, ta **nhảy thẳng** tới trung bình của đoạn "tốt nhất theo λ hiện tại". Hội tụ siêu tuyến tính — thường chỉ vài vòng. Dùng số nguyên: `λ = p / q`, xét `q·a_i - p` để không có sai số.

In [15]:
def best_segment(nums, k, p, q):
    """Đoạn độ dài >= k tối đa hóa Σ(q*a_i - p). Trả về (giá trị, tổng nums, độ dài)."""
    S = [0] + list(accumulate(nums))
    n = len(nums)
    best = None
    min_val, min_idx = None, None
    for j in range(k, n + 1):
        i = j - k
        vi = q * S[i] - p * i                      # tiền tố của b tại i (nhân q)
        if min_val is None or vi < min_val:
            min_val, min_idx = vi, i
        vj = q * S[j] - p * j
        val = vj - min_val
        if best is None or val > best[0]:
            best = (val, S[j] - S[min_idx], j - min_idx)
    return best

def max_avg_dinkelbach(nums, k, verbose=False):
    p, q = sum(nums[:k]), k                        # λ ban đầu: k phần tử đầu
    rounds = 0
    while True:
        rounds += 1
        val, tot, length = best_segment(nums, k, p, q)
        if verbose:
            print(f"vòng {rounds}: λ = {Fraction(p, q)} ≈ {p / q:.5f}; max Σ(q·a - p) = {val}; đoạn tốt nhất: tổng {tot}, dài {length}")
        if val == 0:
            return Fraction(p, q)
        p, q = tot, length

print(max_avg_dinkelbach(nums1, k1, verbose=True))
assert max_avg_dinkelbach(nums1, k1) == Fraction(51, 4)

vòng 1: λ = 1/2 ≈ 0.50000; max Σ(q·a - p) = 208; đoạn tốt nhất: tổng 55, dài 6
vòng 2: λ = 55/6 ≈ 9.16667; max Σ(q·a - p) = 86; đoạn tốt nhất: tổng 51, dài 4
vòng 3: λ = 51/4 ≈ 12.75000; max Σ(q·a - p) = 0; đoạn tốt nhất: tổng 51, dài 4
51/4


**Đọc code:**

- `q * S[i] - p * i` = `q · (S[i] - λ·i)` — tiền tố của `b` nhân `q` để giữ số nguyên.
- `best_segment` giống bài phụ 3.5 nhưng nhớ thêm **vị trí min** để biết đoạn tốt nhất (tổng, độ dài).
- `val >= 0` luôn đúng (đoạn ứng với `λ` hiện tại cho tổng 0). `val > 0` ⇔ có đoạn trung bình lớn hơn `λ` → `λ` tăng nghiêm ngặt.
- Vì chỉ có hữu hạn đoạn và `λ` tăng nghiêm ngặt, thuật toán dừng; thực tế vài vòng.

**Code nộp LeetCode (Cách 4):**

In [16]:
class SolutionDinkelbach:
    def findMaxAverage(self, nums: List[int], k: int) -> float:
        n = len(nums)
        S = [0]
        for x in nums:
            S.append(S[-1] + x)
        p, q = S[k], k
        while True:
            best_val, best_tot, best_len = -1, 0, 1
            min_val, min_idx = None, 0
            for j in range(k, n + 1):
                i = j - k
                vi = q * S[i] - p * i
                if min_val is None or vi < min_val:
                    min_val, min_idx = vi, i
                val = q * S[j] - p * j - min_val
                if val > best_val:
                    best_val, best_tot, best_len = val, S[j] - S[min_idx], j - min_idx
            if best_val == 0:
                return p / q
            p, q = best_tot, best_len

assert SolutionDinkelbach().findMaxAverage(nums1, k1) == 12.75
assert SolutionDinkelbach().findMaxAverage([5], 1) == 5.0
print("SolutionDinkelbach OK")

SolutionDinkelbach OK


**Độ phức tạp Cách 4:** mỗi vòng `O(n)`; số vòng không có cận đẹp trong lý thuyết tổng quát nhưng thực tế rất nhỏ (thường `< 10`). Space `O(n)`.

## 8. TRACE từng bước

### 8.1. Binary search (Cách 2) với ví dụ 1

| vòng | lo | hi | mid | check(mid) |
|---|---|---|---|---|
| 1 | -6 | 50 | 22 | sai → hi = 22 |
| 2 | -6 | 22 | 8 | đúng → lo = 8 |
| 3 | 8 | 22 | 15 | sai → hi = 15 |
| 4 | 8 | 15 | 11.5 | đúng → lo = 11.5 |
| 5 | 11.5 | 15 | 13.25 | sai → hi = 13.25 |
| ... | | | | hội tụ về 12.75 |

In [17]:
def trace_binary(nums, k, eps=1e-6, show=8):
    lo, hi = float(min(nums)), float(max(nums))
    it = 0
    print(f"{'vòng':>4} | {'lo':>12} | {'hi':>12} | {'mid':>12} | check")
    while hi - lo > eps:
        it += 1
        mid = (lo + hi) / 2
        ok = check(nums, k, mid)
        if it <= show:
            print(f"{it:>4} | {lo:>12.6f} | {hi:>12.6f} | {mid:>12.6f} | {'đúng -> lo = mid' if ok else 'sai  -> hi = mid'}")
        if ok:
            lo = mid
        else:
            hi = mid
    print(f"... tổng {it} vòng, kết quả lo = {lo:.7f}")
    return lo

r = trace_binary(nums1, k1)
assert abs(r - 12.75) < 1e-5

vòng |           lo |           hi |          mid | check
   1 |    -6.000000 |    50.000000 |    22.000000 | sai  -> hi = mid
   2 |    -6.000000 |    22.000000 |     8.000000 | đúng -> lo = mid
   3 |     8.000000 |    22.000000 |    15.000000 | sai  -> hi = mid
   4 |     8.000000 |    15.000000 |    11.500000 | đúng -> lo = mid
   5 |    11.500000 |    15.000000 |    13.250000 | sai  -> hi = mid
   6 |    11.500000 |    13.250000 |    12.375000 | đúng -> lo = mid
   7 |    12.375000 |    13.250000 |    12.812500 | sai  -> hi = mid
   8 |    12.375000 |    12.812500 |    12.593750 | đúng -> lo = mid
... tổng 26 vòng, kết quả lo = 12.7499994


### 8.2. Bao lồi (Cách 3): trạng thái hull và điểm tốt nhất cho mỗi `j`

In [18]:
def trace_hull(nums, k):
    S = [0] + list(accumulate(nums))
    n = len(nums)
    hull = []
    best = None
    print("S =", S)
    for j in range(k, n + 1):
        p = j - k
        popped = []
        while len(hull) >= 2:
            a, b = hull[-2], hull[-1]
            if (b - a) * (S[p] - S[a]) - (S[b] - S[a]) * (p - a) <= 0:
                popped.append(hull.pop())
            else:
                break
        hull.append(p)
        cands = [(Fraction(S[j] - S[i], j - i), i) for i in hull]
        val, i = max(cands)
        if best is None or val > best[0]:
            best = (val, i, j)
        print(f"j={j}: thêm P{p}{' (bỏ ' + str(popped) + ')' if popped else ''}; hull={hull}; "
              f"hệ số góc tới P{j}: {[str(c) for c, _ in cands]} -> chọn i={i} ({val})")
    print(f"tốt nhất: đoạn [{best[1]}, {best[2]}) trung bình {best[0]} = {float(best[0])}")
    return best[0]

assert trace_hull(nums1, k1) == Fraction(51, 4)

S = [0, 1, 13, 8, 2, 52, 55]
j=4: thêm P0; hull=[0]; hệ số góc tới P4: ['1/2'] -> chọn i=0 (1/2)
j=5: thêm P1; hull=[0, 1]; hệ số góc tới P5: ['52/5', '51/4'] -> chọn i=1 (51/4)
j=6: thêm P2; hull=[0, 1, 2]; hệ số góc tới P6: ['55/6', '54/5', '21/2'] -> chọn i=1 (54/5)
tốt nhất: đoạn [1, 5) trung bình 51/4 = 12.75


## 9. Code nộp LeetCode

Lời giải chính: **Binary search trên đáp án + kiểm tra `O(n)`** (Cách 2) — đơn giản, chuẩn mực, đủ nhanh. (Muốn kết quả chính xác tuyệt đối thì dùng `SolutionHull` hoặc `SolutionDinkelbach`.)

In [19]:
from typing import List

class Solution:
    def findMaxAverage(self, nums: List[int], k: int) -> float:
        n = len(nums)

        def exists_avg_at_least(x: float) -> bool:
            # Có đoạn độ dài >= k mà tổng (nums[i] - x) >= 0 ?
            cur = 0.0
            for i in range(k):
                cur += nums[i] - x
            if cur >= 0:
                return True
            prev = 0.0          # tiền tố tới j - k
            min_prev = 0.0      # min các tiền tố tới j - k
            for j in range(k, n):
                cur += nums[j] - x
                prev += nums[j - k] - x
                if prev < min_prev:
                    min_prev = prev
                if cur - min_prev >= 0:
                    return True
            return False

        lo, hi = float(min(nums)), float(max(nums))
        while hi - lo > 1e-6:
            mid = (lo + hi) / 2
            if exists_avg_at_least(mid):
                lo = mid
            else:
                hi = mid
        return lo

sol = Solution()
assert abs(sol.findMaxAverage([1, 12, -5, -6, 50, 3], 4) - 12.75) < 1e-5
assert abs(sol.findMaxAverage([5], 1) - 5.0) < 1e-5
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đổi câu hỏi).** Với đoạn có tổng `T`, độ dài `L > 0`: `T / L >= x ⇔ T - x·L >= 0 ⇔ Σ (a_i - x) >= 0`.

**Bổ đề 2 (kiểm tra đúng).** Tồn tại đoạn `[i, j)` với `j - i >= k` và `B[j] - B[i] >= 0` ⇔ tồn tại `j >= k` với `B[j] - min_{i <= j-k} B[i] >= 0`. Hàm `check` duyệt mọi `j` và giữ đúng `min` đó.

**Bổ đề 3 (đơn điệu).** Nếu `check(x)` đúng và `x' < x` thì `check(x')` đúng (cùng đoạn đó). Gọi `x*` là trung bình lớn nhất: `check(x)` đúng ⇔ `x <= x*`.

**Binary search.** Bất biến `lo <= x* <= hi`: ban đầu `min(nums) <= x* <= max(nums)`. `check(mid)` đúng ⇒ `x* >= mid` → `lo = mid`; sai ⇒ `x* < mid` → `hi = mid`. Khi dừng `hi - lo <= 10^-6` nên `|lo - x*| <= 10^-6 < 10^-5`.

**Bao lồi (Cách 3).** Với `j` cố định, cần `max_{i <= j-k}` hệ số góc `P_i P_j`. Nếu `P_i` nằm trên hoặc phía trên đoạn nối hai điểm `P_a, P_b` (`a < i < b <= j - k`) thì `max(slope(P_a,P_j), slope(P_b,P_j)) >= slope(P_i,P_j)` — vì `P_j` nằm bên phải, đường thẳng qua `P_j` "chạm từ dưới" tập điểm luôn chạm tại đỉnh của bao lồi dưới. Trên bao lồi dưới (hệ số góc các cạnh tăng dần), hệ số góc tới `P_j` tăng tới điểm tiếp xúc rồi giảm → binary search tìm được đỉnh.

**Dinkelbach (Cách 4).** Đặt `F(λ) = max Σ(a_i - λ)`. `F(λ) >= 0` luôn đúng khi `λ` là trung bình của một đoạn hợp lệ. Nếu `F(λ) > 0` tại đoạn `D` thì trung bình `D > λ` → `λ` tăng nghiêm ngặt. Nếu `F(λ) = 0` thì không có đoạn nào trung bình `> λ` → `λ = x*`. Hữu hạn đoạn nên dừng.

## 11. Phân tích Time/Space

| Cách | Time | Space | Kết quả |
|---|---|---|---|
| 1. Brute | `O(n^2)` | `O(n)` | chính xác |
| 2. Binary search | `O(n · log(R/ε))` ≈ `34n` | `O(1)` | sai số ≤ `ε` |
| 3. Bao lồi | `O(n log n)` | `O(n)` | chính xác |
| 4. Dinkelbach | `O(n · vòng)` | `O(n)` | chính xác |

In [20]:
random.seed(42)
big = [random.randint(-10**4, 10**4) for _ in range(10**4)]
for kk in (1, 50, 5000):
    row = []
    for name, f in [("binary", Solution().findMaxAverage), ("hull", SolutionHull().findMaxAverage), ("dinkelbach", SolutionDinkelbach().findMaxAverage)]:
        t = time.perf_counter()
        r = f(big, kk)
        row.append(f"{name}={r:.6f} ({time.perf_counter() - t:.3f}s)")
    print(f"k={kk}: " + " | ".join(row))

k=1: binary=9998.999999 (0.002s) | hull=9999.000000 (0.008s) | dinkelbach=9999.000000 (0.013s)


k=50: binary=2616.568627 (0.023s) | hull=2616.568627 (0.009s) | dinkelbach=2616.568627 (0.008s)


k=5000: binary=182.227549 (0.015s) | hull=182.227550 (0.005s) | dinkelbach=182.227550 (0.002s)


## 12. Bộ test

Gồm: ví dụ đề, edge case (`n = 1`, `k = n`, toàn số âm, toàn bằng nhau, giá trị cực trị) và đối chiếu ngẫu nhiên có seed: các cách chính xác phải **bằng đúng** brute force `Fraction`, cách binary search sai số `< 10^-5`.

In [21]:
def check_case(nums, k):
    want = max_avg_brute_exact(nums, k)
    assert max_avg_hull(nums, k) == want, ("hull", nums, k)
    assert max_avg_dinkelbach(nums, k) == want, ("dinkelbach", nums, k)
    for f in (max_avg_binary, SolutionBinary().findMaxAverage, Solution().findMaxAverage,
              SolutionHull().findMaxAverage, SolutionDinkelbach().findMaxAverage, max_avg_brute):
        got = f(nums, k)
        assert abs(got - float(want)) < 1e-5, (f, nums, k, got, float(want))
    return want

fixed = [
    ([1, 12, -5, -6, 50, 3], 4, Fraction(51, 4)),
    ([5], 1, Fraction(5)),
    ([3, 7, 2], 3, Fraction(4)),                  # k = n
    ([-5, -1, -3, -2], 2, Fraction(-2)),          # toàn âm: [-1,-3] hoặc [-1,-3,-2]
    ([4, 4, 4, 4], 2, Fraction(4)),               # toàn bằng nhau
    ([-10**4, 10**4], 1, Fraction(10**4)),        # cực trị
    ([0, 0, 10, 9, 0, 0], 2, Fraction(19, 2)),
    ([1, 2, 3, 4, 5], 1, Fraction(5)),
]
for nums, k, want in fixed:
    assert check_case(nums, k) == want, (nums, k)
print("Tất cả", len(fixed), "test cố định OK")

Tất cả 8 test cố định OK


In [22]:
random.seed(2024)
for trial in range(500):
    n_ = random.randint(1, 12)
    nums = [random.randint(-20, 20) for _ in range(n_)]
    k = random.randint(1, n_)
    check_case(nums, k)
print("500 test ngẫu nhiên: các cách chính xác khớp Fraction, binary search sai số < 1e-5")

500 test ngẫu nhiên: các cách chính xác khớp Fraction, binary search sai số < 1e-5


## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ xét đoạn dài **đúng** `k`

**Code sai:** cửa sổ trượt độ dài `k` (lời giải LC 643).

**Vì sao sai:** đề cho phép độ dài `>= k`; đoạn dài hơn có thể tốt hơn.

In [23]:
def only_len_k_wrong(nums, k):
    s = sum(nums[:k])
    best = s
    for i in range(k, len(nums)):
        s += nums[i] - nums[i - k]
        best = max(best, s)
    return best / k

ex_long2 = [10, 0, 10, 10, 0, 10]
print("sai :", only_len_k_wrong(ex_long2, 3), "| đúng:", float(max_avg_brute_exact(ex_long2, 3)))

sai : 6.666666666666667 | đúng: 7.5


**Code đúng:** xét mọi độ dài `>= k` (Cách 2/3/4).

### Lỗi 2: Lấy `min` tiền tố tới `j` thay vì tới `j - k`

**Code sai:** cập nhật `min_prev` bằng `cur` (tiền tố hiện tại) → cho phép đoạn **ngắn hơn `k`** (kể cả rỗng!).

In [24]:
def check_wrong_min(nums, k, x):
    cur, min_prev = 0.0, 0.0
    for j in range(len(nums)):
        cur += nums[j] - x
        if j + 1 >= k and cur - min_prev >= 0:
            return True
        min_prev = min(min_prev, cur)          # SAI: đoạn có thể ngắn hơn k
    return False

def binary_with(check_fn, nums, k):
    lo, hi = float(min(nums)), float(max(nums))
    while hi - lo > 1e-6:
        mid = (lo + hi) / 2
        if check_fn(nums, k, mid):
            lo = mid
        else:
            hi = mid
    return lo

print("sai :", round(binary_with(check_wrong_min, nums1, k1), 5), "| đúng:", round(binary_with(check, nums1, k1), 5))

sai : 50.0 | đúng: 12.75


**Code đúng:** `min_prev` chỉ gồm tiền tố tới `j - k` (biến `prev` chạy chậm hơn `k` bước).

### Lỗi 3: Điều kiện dừng theo số vòng quá ít hoặc `eps` quá lớn

**Code sai:** `while hi - lo > 1e-3` hoặc chỉ lặp 10 vòng → sai số vượt `10^-5`.

In [25]:
def binary_eps(nums, k, eps):
    lo, hi = float(min(nums)), float(max(nums))
    while hi - lo > eps:
        mid = (lo + hi) / 2
        if check(nums, k, mid):
            lo = mid
        else:
            hi = mid
    return lo

for eps in (1e-2, 1e-3, 1e-6):
    r = binary_eps(nums1, k1, eps)
    print(f"eps={eps:g}: {r:.7f}, sai số {abs(r - 12.75):.2e} {'(đạt)' if abs(r - 12.75) < 1e-5 else '(KHÔNG đạt)'}")

eps=0.01: 12.7441406, sai số 5.86e-03 (KHÔNG đạt)
eps=0.001: 12.7492676, sai số 7.32e-04 (KHÔNG đạt)
eps=1e-06: 12.7499994, sai số 5.96e-07 (đạt)


**Code đúng:** `eps = 1e-6` (nhỏ hơn yêu cầu 10 lần) hoặc lặp cố định ~50 vòng.

### Lỗi 4: So sánh trung bình bằng số thực trong lời giải "chính xác"

Trong bao lồi / Dinkelbach, nếu dùng phép chia số thực để so sánh hệ số góc thì hai giá trị rất gần nhau có thể bị so sai do làm tròn. Luôn **nhân chéo** số nguyên như trong code.

## 14. Mẹo & nhận dạng pattern

- **"Tối đa hóa trung bình / tỉ số"** → thử **binary search trên đáp án** với mẹo "trừ `x` rồi hỏi tổng `>= 0`" (còn gọi là *fractional programming*).
- **"Đoạn con độ dài ít nhất `k`"** → tổng tiền tố + `min` tiền tố trễ `k` bước.
- Trung bình đoạn = **hệ số góc** giữa hai điểm tiền tố → hình học, bao lồi.
- Muốn kết quả chính xác cho bài tỉ số → **Dinkelbach** hoặc so sánh phân số bằng nhân chéo.
- Bài cùng họ: 643 (độ dài cố định), 1150..., "Maximum Density Segment", 2602..., K-th smallest prime fraction (786), Minimum cost to hire K workers (857).

## 15. Biến thể

1. **Độ dài đúng bằng `k`** (LC 643) — cửa sổ trượt `O(n)`.
2. **Độ dài trong khoảng `[k, m]`** — kiểm tra `O(n)` vẫn được nhưng cần **min trên cửa sổ trượt** (deque đơn điệu).
3. **Trả về chỉ số đoạn tối ưu** — Dinkelbach/bao lồi cho sẵn đoạn.
4. **Trung bình có trọng số** `Σa_i / Σw_i` — Dinkelbach tổng quát: xét `a_i - λ·w_i`.

In [26]:
from collections import deque

def max_avg_len_between(nums, k, m, eps=1e-6):
    """Biến thể 2: độ dài trong [k, m]."""
    n = len(nums)
    def ok(x):
        B = [0.0]
        for a in nums:
            B.append(B[-1] + a - x)
        dq = deque()                          # chỉ số i tăng dần, B[i] tăng dần (min ở đầu)
        for j in range(k, n + 1):
            i_new = j - k
            while dq and B[dq[-1]] >= B[i_new]:
                dq.pop()
            dq.append(i_new)
            while dq[0] < j - m:
                dq.popleft()
            if B[j] - B[dq[0]] >= 0:
                return True
        return False
    lo, hi = float(min(nums)), float(max(nums))
    while hi - lo > eps:
        mid = (lo + hi) / 2
        if ok(mid):
            lo = mid
        else:
            hi = mid
    return lo

def brute_between(nums, k, m):
    S = [0] + list(accumulate(nums))
    n = len(nums)
    return max(Fraction(S[j] - S[i], j - i) for i in range(n) for j in range(i + k, min(n, i + m) + 1))

random.seed(3)
for _ in range(200):
    nums = [random.randint(-10, 10) for _ in range(random.randint(1, 10))]
    k = random.randint(1, len(nums)); m = random.randint(k, len(nums))
    assert abs(max_avg_len_between(nums, k, m) - float(brute_between(nums, k, m))) < 1e-5
print("Biến thể 2 OK; ví dụ:", round(max_avg_len_between(nums1, 2, 3), 5), "vs", brute_between(nums1, 2, 3))

Biến thể 2 OK; ví dụ: 26.5 vs 53/2


In [27]:
def max_weighted_avg(a, w, k):
    """Biến thể 4: max Σa/Σw trên đoạn độ dài >= k (w > 0), Dinkelbach chính xác."""
    n = len(a)
    A = [0] + list(accumulate(a))
    W = [0] + list(accumulate(w))
    p, q = A[k], W[k]
    while True:
        best = None
        min_v, min_i = None, 0
        for j in range(k, n + 1):
            i = j - k
            vi = q * A[i] - p * W[i]
            if min_v is None or vi < min_v:
                min_v, min_i = vi, i
            val = q * A[j] - p * W[j] - min_v
            if best is None or val > best[0]:
                best = (val, A[j] - A[min_i], W[j] - W[min_i])
        if best[0] == 0:
            return Fraction(p, q)
        p, q = best[1], best[2]

a, w = [3, 1, 4, 1, 5], [1, 2, 1, 3, 1]
want = max(Fraction(sum(a[i:j]), sum(w[i:j])) for i in range(5) for j in range(i + 2, 6))
print(max_weighted_avg(a, w, 2), want)
assert max_weighted_avg(a, w, 2) == want

2 2


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | mọi đoạn, tổng tiền tố | `O(n^2)` | `O(n)` | trọng tài, `n` nhỏ |
| 2. Binary search | `avg >= x ⇔ Σ(a - x) >= 0` | `O(n log(R/ε))` | `O(1)` | bản nộp chuẩn, dễ nhớ |
| 3. Bao lồi | hệ số góc lớn nhất tới bao lồi dưới | `O(n log n)` | `O(n)` | cần chính xác, thích hình học |
| 4. Dinkelbach | nhảy `λ` tới trung bình đoạn tốt nhất | `O(n · vòng)` | `O(n)` | bài tỉ số tổng quát, chính xác |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `findMaxAverage([0, 4, 0, 3, 2], 1)`, `findMaxAverage([0, 4, 0, 3, 2], 2)`, `findMaxAverage([-1, -2, -3], 3)`?

**Bài 2 (tính tay).** Tính tay `check([0, 4, 0, 3, 2], 2, 2.5)` theo bảng như mục 4.3.

**Bài 3 (điền TODO).** Hoàn thiện `my_check(nums, k, x)` (không nhìn code).

**Bài 4 (tự viết lại).** Viết lại toàn bộ Cách 2 `my_find_max_average(nums, k)`.

**Bài 5 (mini challenge).** Viết `best_subarray(nums, k)` trả về `(trung bình chính xác dạng Fraction, l, r)` với đoạn `nums[l..r]` (bao gồm hai đầu) đạt trung bình lớn nhất và độ dài `>= k`.

In [28]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def my_check(nums, k, x):
    # TODO: cur (tiền tố tới j), prev (tiền tố tới j-k), min_prev
    pass

# Bài 4 — TODO
def my_find_max_average(nums, k):
    # TODO: lo = min, hi = max; while hi - lo > 1e-6
    pass

# Bài 5 — TODO
def best_subarray(nums, k):
    # TODO: Dinkelbach, nhớ thêm vị trí min_idx và j của đoạn tốt nhất
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `k = 1` thì đoạn tốt nhất là phần tử lớn nhất.
- Mức 2: `k = 2`: thử `[4,0]` (2), `[0,3,2]`? dài 3 → 5/3; `[4,0,3]` → 7/3; `[3,2]` → 2.5.
- Mức 3: đáp án `4`, `2.5`, `-2`.

**Bài 2.**
- Mức 1: `b = [-2.5, 1.5, -2.5, 0.5, -0.5]`.
- Mức 2: `B = [0, -2.5, -1, -3.5, -3, -3.5]`.
- Mức 3: `j = 2`: `-1 - 0 < 0`; `j = 3`: `-3.5 - min(0,-2.5) = -1`; `j = 4`: `-3 - (-2.5) = -0.5`; `j = 5`: `-3.5 - (-3.5) = 0 >= 0` → đúng.

**Bài 3.**
- Mức 1: `cur = Σ_{i<k} (nums[i] - x)`.
- Mức 2: vòng `j` từ `k`: `cur += nums[j] - x`, `prev += nums[j-k] - x`.
- Mức 3: `min_prev = min(min_prev, prev)`; `if cur >= min_prev: return True`.

**Bài 4.**
- Mức 1: dùng `my_check`.
- Mức 2: `lo = mid` khi đúng.
- Mức 3: trả `lo`.

**Bài 5.**
- Mức 1: dựa trên `best_segment`.
- Mức 2: lưu thêm `min_idx` và `j` khi cập nhật `best`.
- Mức 3: đoạn là `nums[min_idx .. j-1]`.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert abs(Solution().findMaxAverage([0, 4, 0, 3, 2], 1) - 4) < 1e-5
assert abs(Solution().findMaxAverage([0, 4, 0, 3, 2], 2) - 2.5) < 1e-5
assert abs(Solution().findMaxAverage([-1, -2, -3], 3) + 2) < 1e-5

# Bài 2
assert check_verbose([0, 4, 0, 3, 2], 2, 2.5) is True

# Bài 3
def my_check_ans(nums, k, x):
    cur = sum(nums[:k]) - x * k
    if cur >= 0:
        return True
    prev = min_prev = 0.0
    for j in range(k, len(nums)):
        cur += nums[j] - x
        prev += nums[j - k] - x
        min_prev = min(min_prev, prev)
        if cur >= min_prev:
            return True
    return False

random.seed(12)
for _ in range(300):
    nums = [random.randint(-9, 9) for _ in range(random.randint(1, 8))]
    k = random.randint(1, len(nums))
    x = random.uniform(-10, 10)
    assert my_check_ans(nums, k, x) == check(nums, k, x)
print("Bài 1–3 OK")

x = 2.5: b = [-2.5, 1.5, -2.5, 0.5, -0.5]
  j=2: B[j]=  -1.0, min tiền tố=     0, hiệu=  -1.0
  j=3: B[j]=  -3.5, min tiền tố=  -2.5, hiệu=  -1.0
  j=4: B[j]=  -3.0, min tiền tố=  -2.5, hiệu=  -0.5
  j=5: B[j]=  -3.5, min tiền tố=  -3.5, hiệu=   0.0
Bài 1–3 OK


In [30]:
# Bài 4
def my_find_max_average_ans(nums, k):
    lo, hi = float(min(nums)), float(max(nums))
    while hi - lo > 1e-6:
        mid = (lo + hi) / 2
        if my_check_ans(nums, k, mid):
            lo = mid
        else:
            hi = mid
    return lo

# Bài 5
def best_subarray_ans(nums, k):
    n = len(nums)
    S = [0] + list(accumulate(nums))
    p, q = S[k], k
    seg = (0, k - 1)
    while True:
        best = None
        min_v, min_i = None, 0
        for j in range(k, n + 1):
            i = j - k
            vi = q * S[i] - p * i
            if min_v is None or vi < min_v:
                min_v, min_i = vi, i
            val = q * S[j] - p * j - min_v
            if best is None or val > best[0]:
                best = (val, min_i, j)
        if best[0] == 0:
            return Fraction(p, q), seg[0], seg[1]
        _, i, j = best
        p, q, seg = S[j] - S[i], j - i, (i, j - 1)

print(best_subarray_ans(nums1, k1))
random.seed(13)
for _ in range(300):
    nums = [random.randint(-15, 15) for _ in range(random.randint(1, 10))]
    k = random.randint(1, len(nums))
    want = max_avg_brute_exact(nums, k)
    assert abs(my_find_max_average_ans(nums, k) - float(want)) < 1e-5
    val, l, r = best_subarray_ans(nums, k)
    assert val == want and r - l + 1 >= k and Fraction(sum(nums[l:r + 1]), r - l + 1) == want
print("Bài 4–5 OK (đối chiếu brute force Fraction)")

(Fraction(51, 4), 1, 4)
Bài 4–5 OK (đối chiếu brute force Fraction)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "tối đa hóa trung bình, độ dài >= k":**

```python
def ok(x):
    cur = sum(nums[:k]) - x * k
    if cur >= 0: return True
    prev = min_prev = 0.0
    for j in range(k, n):
        cur += nums[j] - x
        prev += nums[j - k] - x
        min_prev = min(min_prev, prev)
        if cur >= min_prev: return True
    return False

lo, hi = min(nums), max(nums)
while hi - lo > 1e-6:
    mid = (lo + hi) / 2
    if ok(mid): lo = mid
    else: hi = mid
return lo
```

**Checklist:**

- [ ] Độ dài **ít nhất** `k`, không phải đúng `k`.
- [ ] `min_prev` chỉ gồm tiền tố tới `j - k` (khởi tạo `0` = tiền tố rỗng).
- [ ] Khoảng tìm kiếm `[min(nums), max(nums)]`.
- [ ] `eps` nhỏ hơn yêu cầu (`1e-6` cho yêu cầu `1e-5`).
- [ ] Lời giải chính xác: so sánh phân số bằng **nhân chéo** số nguyên.
- [ ] Thử `n = 1`, `k = n`, toàn âm, toàn bằng nhau.